# GC Recent Intrabar Diagnostic

This notebook is a recent-data diagnostic workflow for manual out-of-sample intrabar inspection. It is intentionally separate from the main historical backtest.

Scope limits:

- Data source is `yfinance` (`GC=F`), not the main historical futures dataset.
- Coverage is only the recent 7-day intraday window allowed by the source.
- The replay logic is diagnostic-only and does not redefine the production backtest.
- Independent long/short mode is available here for recent path inspection only.


## 0. Configuration

The defaults are cache-first. Set `force_refresh = True` only when you want to download the latest recent sample from `yfinance`.


In [8]:
from pathlib import Path

from IPython.display import display

from gold_breakout_intrabar_recent import (
    RecentIntrabarConfig,
    RecentIntrabarReplayConfig,
    recent_intrabar_config_frame,
    recent_intrabar_replay_config_frame,
)

config = RecentIntrabarConfig(
    ticker="GC=F",
    period="7d",
    interval="1m",
    cache_dir=Path("cache") / "yfinance_intrabar_recent",
    session_gap_threshold_minutes=30,
)

replay_config = RecentIntrabarReplayConfig(
    tp_sl_mode="prior_day_range",
    fixed_ticks=100,
    atr_multiplier_tpsl=2.0,
    k=0.50,
    atr_period=14,
    atr_multiplier=1.5,
    allow_independent_long_short=True,
    close_positions_at_session_end=True,
    frame_step=5,
    animation_interval_ms=120,
)

display(recent_intrabar_config_frame(config))
display(recent_intrabar_replay_config_frame(replay_config))


ImportError: cannot import name 'RecentIntrabarReplayConfig' from 'gold_breakout_intrabar_recent' (c:\Users\Luth Haidar\Downloads\gold_break\gold_breakout_intrabar_recent.py)

## 1. Imports

The helper module handles cache loading, recent session reconstruction, diagnostic simulation, and replay plotting.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

from gold_breakout_intrabar_recent import (
    animate_intrabar_session_replay,
    build_recent_intrabar_context,
    plot_intrabar_session_replay,
    recent_intrabar_cache_paths,
    simulate_intrabar_diagnostic,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 120


## 2. Load Cached Or Recent Data

This keeps the workflow reproducible and avoids unnecessary downloads during replay work.


In [ ]:
force_refresh = False
context = build_recent_intrabar_context(config, force_refresh=force_refresh)
cache_paths = recent_intrabar_cache_paths(config)

display(pd.DataFrame([context["metadata"]]))
display(pd.DataFrame({"cache_key": list(cache_paths.keys()), "path": [str(path) for path in cache_paths.values()]}))
display(context["session_table"])


## 3. Raw Session Preview

These tables make the recent sample auditable before any replay logic runs.


In [ ]:
display(context["minute_bars"].head(10))
display(context["hourly_bars"].head(10))


## 4. Run Diagnostic Intrabar Simulation

This simulation is minute-based and diagnostic-only. The default here allows independent long/short entries so ambiguous recent path behavior can be inspected directly.


In [ ]:
diagnostic = simulate_intrabar_diagnostic(context, replay_config)

display(diagnostic["session_summary"])
display(diagnostic["trade_log"].head(20))
display(diagnostic["event_log"].head(20))
display(diagnostic["frame_state"].head(20))


## 5. Select A Replay Session

Pick the first recent session with at least one closed trade, falling back to the first simulated session if needed.


In [ ]:
if diagnostic["session_summary"].empty:
    raise ValueError("No simulated sessions are available for replay.")

trade_sessions = diagnostic["session_summary"].loc[
    diagnostic["session_summary"]["trade_count"].gt(0),
    "session_id",
]
replay_session_id = (
    int(trade_sessions.iloc[0])
    if not trade_sessions.empty
    else int(diagnostic["session_summary"]["session_id"].iloc[0])
)

replay_session_id


## 6. Static Session Replay

This plot shows the full minute-by-minute replay state for one recent session.


In [ ]:
figure = plot_intrabar_session_replay(diagnostic, replay_session_id)
display(figure)
plt.close(figure)


## 7. Animated Session Replay

The animation reveals the intrabar path progressively. This is intended for manual inspection, not performance reporting.


In [ ]:
animation = animate_intrabar_session_replay(diagnostic, replay_session_id)
HTML(animation.to_jshtml())
